In [1]:
import sys
sys.path.append("..")

In [2]:
# Custom Libraries
from src import load_dataset
from src.preprocessing import show_missing_entries
import src.feature_engineering as feat
from src.imputation import ImputeByGroup, FillCabin, FillNA
from src.log import save_load_model_hist

# Essentials
import numpy as np
import pandas as pd

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold

# Modeling
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Notebook Configs

In [3]:
SEED = 42
DEBUGGING_MODE = True
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [4]:
train_set, test_set = load_dataset()

# Pipeline

## Imputer

In [5]:
imputer_pipeline = Pipeline([
    ('impute_fare', ImputeByGroup(group_by_col = ['Pclass'], col_to_impute = 'Fare', strategy = 'mean')),
    ('impute_age', ImputeByGroup(group_by_col = ['Pclass', 'Sex'], col_to_impute = 'Age', strategy = 'median')),
    ('impute_cabin', FillCabin()),
    ('impute_cabin_embarked', FillNA())
])

## Feature Engineering

In [6]:
feature_pipeline = Pipeline([
     ("Family_size", feat.FeatFamilySize()),
     ("Is_solo", feat.FeatIsSolo()),
     ("Deck", feat.FeatDeck()),
     ("Title", feat.FeatTitle()),
    
     #("Fare_bucket", feat.FeatFareBucket()),
     #("Age_bucket", feat.FeatAgeBucket()),
     #("Deck_missing", feat.FeatDeckMissing()),
     #("Is_ticket_num", feat.FeatIsTicketNum()),
     #("Ticket_char_0", feat.FeatTicketFirstChar()),
     ("Is_Adult", feat.FeatIsAdult()),
     ("Interactions", feat.FeatInteraction())
])

## Drop UNENCODED columns

In [7]:
cols_to_drop = ['PassengerId','Name', 'Ticket', 'Cabin', 'SibSp', 'Parch']

drop_pipeline = Pipeline([
    ('drop', feat.DropColumns(cols_to_drop = cols_to_drop)) 
])

## Encode

In [8]:
col_type_to_encode = ['object', 'category', 'str']
one_hot_encode = (ColumnTransformer([
    (
        'hot_encode', 
         OneHotEncoder(sparse_output=False, handle_unknown='ignore'), 
         make_column_selector(dtype_include=col_type_to_encode)
    )
], remainder = "passthrough", verbose_feature_names_out=False)).set_output(transform='pandas')

## Drop ENCODED columns

### TODO

## Orchestration Pipeline

In [9]:
final_preprocess_pipeline = Pipeline([
    ('impute', imputer_pipeline),
    ('feature', feature_pipeline),
    ('drop_unencoded', drop_pipeline),
    ('encode', one_hot_encode)
])

In [10]:
no_encode_preprocess_pipeline = Pipeline([
    ('impute', imputer_pipeline),
    ('feature', feature_pipeline),
    ('drop_unencoded', drop_pipeline)
])

## Common Description

In [11]:
df = train_set.copy()
transformed_df = no_encode_preprocess_pipeline.fit_transform(df)
final_cols = no_encode_preprocess_pipeline.get_feature_names_out()
cols_to_encode = transformed_df.select_dtypes(include=col_type_to_encode).columns.to_list()
impute_desc = no_encode_preprocess_pipeline.named_steps['impute']

In [12]:
desc = f"""Impute:
{impute_desc}

Final Transformed Features:
{final_cols}

Encoded Columns:
{cols_to_encode}

Scaler:
StandardScaler() for non-tree models
"""

In [13]:
print(desc)

Impute:
Pipeline(steps=[('impute_fare',
                 ImputeByGroup(col_to_impute='Fare', group_by_col=['Pclass'])),
                ('impute_age',
                 ImputeByGroup(col_to_impute='Age',
                               group_by_col=['Pclass', 'Sex'],
                               strategy='median')),
                ('impute_cabin', FillCabin()),
                ('impute_cabin_embarked', FillNA())])

Final Transformed Features:
['Survived' 'Pclass' 'Sex' 'Age' 'Fare' 'Embarked' 'Family_size' 'Is_solo'
 'Deck' 'Title' 'Is_adult' 'Class_1_master' 'Class_3_male']

Encoded Columns:
['Sex', 'Embarked', 'Deck', 'Title']

Scaler:
StandardScaler() for non-tree models



# Modeling

In [14]:
df_train = train_set.copy()
X = df_train.drop(labels = ['Survived'], axis = 1)
y = df_train['Survived']

## Logistic Regression

In [15]:
# Build final pipeline for model
lr_model = Pipeline([
    ('preproces', final_preprocess_pipeline),
    ('scale', StandardScaler().set_output(transform="pandas")),
    ('predictor', LogisticRegression(max_iter = 1000, random_state = SEED, C = 0.1, solver ='lbfgs'))
])

# Set final description
new_desc = desc + f"""
Model:
{lr_model.named_steps['predictor']}
"""

# Cross validation score
scores = cross_val_score(lr_model, X, y, cv=SKF)
print(scores.mean(), scores.std())

# Save model log
latest, last_algo_stat, model_hist = save_load_model_hist('LR', cv_scores=scores, description=desc, clear_model_history = False)

# Show stats for the model
print("Latest", "\n", latest[['Model', 'Mean', 'Std']])
print("Former", "\n", last_algo_stat[['Model', 'Mean', 'Std']])
model_hist[['Model', 'Mean', 'Std']].sort_values(by='Mean', ascending=False).head(5)

0.8406189190885694 0.020966507105340516
Latest 
   Model      Mean       Std
0  V_33  0.840619  0.020967
Former 
   Model      Mean       Std
6   V_7  0.840619  0.020967


,Model,Mean,Std
3,V_4,0.840619,0.020967
0,V_33,0.840619,0.020967
6,V_7,0.840619,0.020967
8,V_9,0.840619,0.020967
11,V_12,0.840619,0.020967


## SVM (rbf)

In [21]:
# Build final pipeline for model
svm_rbf_model = Pipeline([
    ('preproces', final_preprocess_pipeline),
    ('scale', StandardScaler()),
    ('predictor', CalibratedClassifierCV(
        svm.SVC(kernel='rbf', random_state=SEED, C=1),
        ensemble=False))
])

# Set final description
new_desc = desc + f"""
Model:
{svm_rbf_model.named_steps['predictor']}
"""

# Cross validation score
scores = cross_val_score(svm_rbf_model, X, y, cv=SKF)

# Save model log
latest, last_algo_stat, model_hist = save_load_model_hist('SVM rbf', cv_scores=scores, description=desc, clear_model_history = False)

# Show stats for the model
print("Latest", "\n", latest[['Model', 'Mean', 'Std']])
print("Former", "\n", last_algo_stat[['Model', 'Mean', 'Std']])
model_hist[['Model', 'Mean', 'Std']].sort_values(by='Mean', ascending=False).head(5)

Latest 
   Model      Mean       Std
0  V_39  0.832747  0.014633
Former 
    Model      Mean       Std
33  V_34  0.832747  0.014633


,Model,Mean,Std
3,V_4,0.840619,0.020967
6,V_7,0.840619,0.020967
8,V_9,0.840619,0.020967
11,V_12,0.840619,0.020967
32,V_33,0.840619,0.020967


## SVM (linear)

In [17]:
# Build final pipeline for model
svm_lin_model = Pipeline([
    ('preproces', final_preprocess_pipeline),
    ('scale', StandardScaler()),
    ('predictor', CalibratedClassifierCV(
        svm.SVC(kernel='linear', random_state=SEED, C=0.1),
        ensemble=False))
])

# Set final description
new_desc = desc + f"""
Model:
{svm_lin_model.named_steps['predictor']}
"""
# Cross validation score
scores = cross_val_score(svm_lin_model, X, y, cv=SKF)

# Save model log
latest, last_algo_stat, model_hist = save_load_model_hist('SVM lin', cv_scores=scores, description=desc, clear_model_history = False)

# Show stats for the model
print("Latest", "\n", latest[['Model', 'Mean', 'Std']])
print("Former", "\n", last_algo_stat[['Model', 'Mean', 'Std']])
model_hist[['Model', 'Mean', 'Std']].sort_values(by='Mean', ascending=False).head(5)

Latest 
   Model      Mean       Std
0  V_35  0.826031  0.013843
Former 
    Model      Mean       Std
18  V_19  0.824907  0.015329


,Model,Mean,Std
8,V_9,0.840619,0.020967
6,V_7,0.840619,0.020967
3,V_4,0.840619,0.020967
32,V_33,0.840619,0.020967
11,V_12,0.840619,0.020967


## Random Forest

In [18]:
# Build final pipeline for model
rf_model = Pipeline([
    ('preproces', final_preprocess_pipeline),
    ('predictor', RandomForestClassifier(random_state=SEED, n_estimators=90, criterion='entropy', min_samples_split=20, max_depth = 15, min_samples_leaf=1, bootstrap=True, oob_score=True))
])

# Set final description
new_desc = desc + f"""
Model:
{rf_model.named_steps['predictor']}
"""
# Cross validation score
scores = cross_val_score(rf_model, X, y, cv=SKF)

# Save model log
latest, last_algo_stat, model_hist = save_load_model_hist('RF', cv_scores=scores, description=desc, clear_model_history = False)

# Show stats for the model
print("Latest", "\n", latest[['Model', 'Mean', 'Std']])
print("Former", "\n", last_algo_stat[['Model', 'Mean', 'Std']])
model_hist[['Model', 'Mean', 'Std']].sort_values(by='Mean', ascending=False).head(5)

Latest 
   Model    Mean       Std
0  V_36  0.8305  0.015082
Former 
    Model      Mean       Std
30  V_31  0.835007  0.011139


,Model,Mean,Std
3,V_4,0.840619,0.020967
11,V_12,0.840619,0.020967
8,V_9,0.840619,0.020967
6,V_7,0.840619,0.020967
32,V_33,0.840619,0.020967


## XGBoost

In [19]:
# Build final pipeline for model
xgb_model = Pipeline([
    ('preproces', final_preprocess_pipeline),
    ('predictor', XGBClassifier(random_state=SEED, n_estimators=90, max_depth = 3, learning_rate=0.2, subsample=0.3, colsample_bytree = 0.2))
])

# Set final description
new_desc = desc + f"""
Model:
{rf_model.named_steps['predictor']}
"""
# Cross validation score
scores = cross_val_score(xgb_model, X, y, cv=SKF)

# Save model log
latest, last_algo_stat, model_hist = save_load_model_hist('XGB', cv_scores=scores, description=desc, clear_model_history = False)

# Show stats for the model
print("Latest", "\n", latest[['Model', 'Mean', 'Std']])
print("Former", "\n", last_algo_stat[['Model', 'Mean', 'Std']])
model_hist[['Model', 'Mean', 'Std']].sort_values(by='Mean', ascending=False).head(5)

Latest 
   Model      Mean       Std
0  V_37  0.830525  0.006595
Former 
    Model      Mean       Std
31  V_32  0.838378  0.015297


,Model,Mean,Std
3,V_4,0.840619,0.020967
32,V_33,0.840619,0.020967
11,V_12,0.840619,0.020967
8,V_9,0.840619,0.020967
6,V_7,0.840619,0.020967
